## setup

In [1]:
# import stuff
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# load data from folder
folder = Path('../data')

# filtered dataset with mortgage rates
listings = pd.read_csv(folder / 'listings_with_rates.csv', low_memory = False)

# null count summary as reference for cleaning
listings_null_summary = pd.read_csv(folder / 'listings_null_summary.csv', index_col = 0)

In [3]:
print('Listings dataset shape:', listings.shape)
listings.head()

Listings dataset shape: (607724, 86)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,NaN,NaN,90067,NaN,2105.00,177861.0,NaN,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,NaN,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,NaN,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,NaN,317 E. Bayfront,2024-01,6.6425


In [4]:
listings_null_summary.head()

,null ct,null pct
OriginalListPrice,881,0.144967
ListingKey,0,0.000000
ListAgentEmail,114542,18.847701
CloseDate,410502,67.547439
ClosePrice,431906,71.069433


## data cleaning & prep
- convert date fields to datetime format
- remove unnecessary/redundant cols
- handle missing values appropriately
- ensure numeric fields are properly typed
- remove/flag invalid numeric values

In [5]:
# convert date columns to datetime format
date_cols = ['CloseDate',
             'PurchaseContractDate',
             'ListingContractDate',
             'ContractStatusChangeDate']
listings[date_cols] = listings[date_cols].apply(pd.to_datetime, errors = 'coerce')

# check that changes have been made
listings[date_cols].dtypes

CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

In [6]:
# check cols w >90% nulls
flag_over_90 = listings_null_summary[listings_null_summary['null pct'] > 90].index.tolist()
flag_over_90.sort()
print(len(flag_over_90))
flag_over_90

13


['AboveGradeFinishedArea',
 'BelowGradeFinishedArea',
 'BuilderName',
 'BuildingAreaTotal',
 'BusinessType',
 'CoBuyerAgentFirstName',
 'CoveredSpaces',
 'ElementarySchoolDistrict',
 'FireplacesTotal',
 'LotSizeDimensions',
 'MiddleOrJuniorSchoolDistrict',
 'TaxAnnualAmount',
 'TaxYear']

From the ``Real_Estate_Primer.pdf``, recall that our key data fields are:
- ``ListingKey``
- ``ListingContractDate``
- ``ListPrice``
- ``ClosePrice``
- ``PurchaseContractDate``
- ``CloseDate``
- ``LivingArea``
- ``BedroomsTotal``
- ``BathroomsTotalInteger``
- ``Latitude``/``Longitude``
- ``UnparsedAddress``

Since the above columns are not part of key fields, we can remove them.

In [7]:
# drop cols w >90% nulls
listings_clean = listings.drop(columns = flag_over_90)

print('Listings shape after dropping:', listings_clean.shape)
listings_clean.head()

Listings shape after dropping: (607724, 73)


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaT,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaT,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.00,5300.0,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaT,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaT,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,...,False,4.0,Walnut Valley Unified,91765,NaN,295.95,58232.0,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaT,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,...,False,2.0,Newport Mesa Unified,92662,NaN,0.00,2250.0,317 E. Bayfront,2024-01,6.6425


### shape after dropping >90% null: (607724, 73)

We can also consider dropping columns with over 50% nulls for more meaningful analyses, but we will still make sure that none of these flagged columns are considered key data fields. From our Week 2-3 deliverable instructions, we were given a list of key numeric fields (in which I will define as ``core_fields``), so we should also make sure that they are not flagged.

In [8]:
# consider dropping cols w >50% nulls
flag_over_50 = listings_null_summary[listings_null_summary['null pct'] > 50].index.tolist()

# recall wk2-3: remove core fields from the list of cols to drop
core_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
               'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
               'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

for field in core_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields)') # overlaps with >90% null cols
flag_over_50

31 columns with over 50% nulls (excl. core fields)


['AboveGradeFinishedArea',
 'AssociationFeeFrequency',
 'BelowGradeFinishedArea',
 'BuilderName',
 'BuildingAreaTotal',
 'BusinessType',
 'BuyerAgencyCompensation',
 'BuyerAgencyCompensationType',
 'BuyerAgentFirstName',
 'BuyerAgentLastName',
 'BuyerAgentMlsId',
 'BuyerOfficeAOR',
 'BuyerOfficeName',
 'BuyerOfficeName.1',
 'CloseDate',
 'CloseDate.1',
 'CoBuyerAgentFirstName',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'CoListOfficeName',
 'CoveredSpaces',
 'ElementarySchool',
 'ElementarySchoolDistrict',
 'FireplacesTotal',
 'HighSchool',
 'LotSizeDimensions',
 'MiddleOrJuniorSchool',
 'MiddleOrJuniorSchoolDistrict',
 'SubdivisionName',
 'TaxAnnualAmount',
 'TaxYear']

In week 6, we will be feature engineering using existing columns and also adding school districts using properties' latitude and longitude values, so we will exclude removing schools and school districts in case we end up populating them in future deliverables.

In [9]:
# remove schools and school districts from flagged cols
school_fields = ['ElementarySchool',
                 'ElementarySchoolDistrict',
                 'MiddleOrJuniorSchool',
                 'MiddleOrJuniorSchoolDistrict',
                 'HighSchool']

for field in school_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

for i in flag_over_90:
    for j in flag_over_50:
        if j in flag_over_90:
            flag_over_50.remove(j)

flag_over_50.sort()
print(len(flag_over_50), 'columns with over 50% nulls (excl. core fields & schools):')
flag_over_50

15 columns with over 50% nulls (excl. core fields & schools):


['AssociationFeeFrequency',
 'BuyerAgencyCompensation',
 'BuyerAgencyCompensationType',
 'BuyerAgentFirstName',
 'BuyerAgentLastName',
 'BuyerAgentMlsId',
 'BuyerOfficeAOR',
 'BuyerOfficeName',
 'BuyerOfficeName.1',
 'CloseDate',
 'CloseDate.1',
 'CoListAgentFirstName',
 'CoListAgentLastName',
 'CoListOfficeName',
 'SubdivisionName']

In [10]:
# drop cols w >50% nulls (excl. core fields and schools)
listings_clean = listings_clean.drop(columns = flag_over_50)
print('Listings shape after dropping:', listings_clean.shape)

listings_clean.head()

Listings shape after dropping: (607724, 58)


,OriginalListPrice,ListingKey,ListAgentEmail,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,PropertyType,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,UnparsedAddress.1,year_month,rate_30yr_fixed
0,1340000.0,1074973329,haleh360@Gmail.com,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,Residential,...,NaN,False,NaN,NaN,90067,2105.00,177861.0,2220 Avenue Of The Stars 2704,2024-01,6.6425
1,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,Residential,...,0.0,False,3.0,Capistrano Unified,92677,254.00,5300.0,16 Palisades,2024-01,6.6425
2,3150000.0,1074936537,anader@dppre.com,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,Residential,...,NaN,NaN,2.0,NaN,91108,NaN,9404.0,1615 Waverly Road,2024-01,6.6425
3,3090000.0,1074917818,QIANYU0607@GMAIL.COM,NaN,QIANYU,GUAN,33.984057,-117.802819,2250 Indian Creek Road,Residential,...,1.0,False,4.0,Walnut Valley Unified,91765,295.95,58232.0,2250 Indian Creek Road,2024-01,6.6425
4,12725000.0,1074143166,jeff.williams@pacificsir.com,NaN,Jeff,Williams,33.607583,-117.887743,317 E. Bayfront,Residential,...,3.0,False,2.0,Newport Mesa Unified,92662,0.00,2250.0,317 E. Bayfront,2024-01,6.6425


### shape after dropping >50% null: (607724, 58)

We can take a look at the remaining columns and determine what else to remove.

In [11]:
sorted(listings_clean.columns)

['AssociationFee',
 'AttachedGarageYN',
 'BathroomsTotalInteger',
 'BedroomsTotal',
 'City',
 'ClosePrice',
 'ContractStatusChangeDate',
 'CountyOrParish',
 'DaysOnMarket',
 'DaysOnMarket.1',
 'ElementarySchool',
 'FireplaceYN',
 'GarageSpaces',
 'HighSchool',
 'HighSchoolDistrict',
 'Latitude',
 'Latitude.1',
 'Levels',
 'ListAgentEmail',
 'ListAgentFirstName',
 'ListAgentFirstName.1',
 'ListAgentFullName',
 'ListAgentLastName',
 'ListAgentLastName.1',
 'ListOfficeName',
 'ListPrice',
 'ListPrice.1',
 'ListingContractDate',
 'ListingId',
 'ListingKey',
 'ListingKeyNumeric',
 'LivingArea',
 'LivingArea.1',
 'Longitude',
 'Longitude.1',
 'LotSizeAcres',
 'LotSizeArea',
 'LotSizeSquareFeet',
 'MLSAreaMajor',
 'MainLevelBedrooms',
 'MiddleOrJuniorSchool',
 'MlsStatus',
 'NewConstructionYN',
 'OriginalListPrice',
 'ParkingTotal',
 'PostalCode',
 'PropertySubType',
 'PropertyType',
 'PropertyType.1',
 'PurchaseContractDate',
 'StateOrProvince',
 'Stories',
 'StreetNumberNumeric',
 'Unparsed

In [12]:
# set up list to remove remaining cols
cols_to_remove = []

listings[['ListAgentFirstName', 'ListAgentFullName', 'ListAgentLastName', 'ListAgentEmail']].isna().sum()

ListAgentFirstName      4661
ListAgentFullName        101
ListAgentLastName         39
ListAgentEmail        114542
dtype: int64

In [13]:
# there seems to be duplicate cols, so we should remove those too
dupe_cols = [col for col in listings_clean.columns if col.endswith('.1')]
print(dupe_cols)

for col in dupe_cols:
    cols_to_remove.append(col)

cols_to_remove

['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'UnparsedAddress.1']


['PropertyType.1',
 'ListAgentFirstName.1',
 'DaysOnMarket.1',
 'LivingArea.1',
 'Longitude.1',
 'Latitude.1',
 'ListPrice.1',
 'ListAgentLastName.1',
 'UnparsedAddress.1']

In [14]:
# remove first & last name since there's already fullname col
cols_to_remove.append('ListAgentFirstName')
cols_to_remove.append('ListAgentLastName')

# not needed for analysis
cols_to_remove.append('ListAgentEmail')

In [15]:
# street numbers are unique + not needed for analysis
print(listings['StreetNumberNumeric'].value_counts())

cols_to_remove.append('StreetNumberNumeric')

StreetNumberNumeric
1.0        827
100.0      619
10.0       619
15.0       604
6.0        593
          ... 
65770.0      1
48880.0      1
80770.0      1
98830.0      1
44969.0      1
Name: count, Length: 52052, dtype: int64


In [16]:
print(listings_clean['ListingKeyNumeric'].equals(listings_clean['ListingKey']))

cols_to_remove.append('ListingKeyNumeric')

True


In [17]:
print(listings_clean[['PropertyType', 'PropertySubType']].value_counts())

cols_to_remove.append('PropertyType')

PropertyType  PropertySubType      
Residential   SingleFamilyResidence    443490
              Condominium              109554
              Townhouse                 35465
              ManufacturedOnLand         8425
              Duplex                     3851
              StockCooperative           1920
              Cabin                      1127
              Triplex                     775
              MixedUse                    564
              Quadruplex                  321
              MobileHome                  180
              BoatSlip                    146
              OwnYourOwn                  108
              ManufacturedHome            105
              CoOwnership                  82
              Farm                         70
              Loft                         68
              Timeshare                    52
              Studio                       41
              DeededParking                 7
              Apartment                     

In [18]:
# print(listings_clean[['LotSizeAcres', 'LotSizeArea', 'LotSizeSquareFeet']])
print(listings_clean['LotSizeArea'].equals(listings_clean['LotSizeSquareFeet']))

cols_to_remove.append('LotSizeArea')

False


In [19]:
print(listings_clean['StateOrProvince'].unique())

cols_to_remove.append('StateOrProvince')

['CA' 'AZ' 'TX' nan 'OS' 'FL' 'GA' 'MD' 'SC' 'CO' 'NY' 'ME' 'MO' 'NV' 'OR'
 'BC' 'WA' 'TN' 'DC' 'HI' 'PA' 'IL' 'MT' 'LA' 'NJ' 'VA' 'NC' 'AL' 'KS']


### REMOVE COLUMNS

In [20]:
# double check before removing
cols_to_remove

['PropertyType.1',
 'ListAgentFirstName.1',
 'DaysOnMarket.1',
 'LivingArea.1',
 'Longitude.1',
 'Latitude.1',
 'ListPrice.1',
 'ListAgentLastName.1',
 'UnparsedAddress.1',
 'ListAgentFirstName',
 'ListAgentLastName',
 'ListAgentEmail',
 'StreetNumberNumeric',
 'ListingKeyNumeric',
 'PropertyType',
 'LotSizeArea',
 'StateOrProvince']

#### reviewing columns to remove

| column name | description |
| ----------- | ---------- |
| ListAgentFirstName | The first name of the listing agent |
| ListAgentLastName | The last name of the listing agent |
| StreetNumberNumeric | The integer portion of the street number |
| PropertyType | The list of types of properties such as Residential, Lease, Income, Land, Mobile, Commercial Sale, etc |
| ListAgentEmail | The email address of the Listing Agent |
| LotSizeArea | The total area of the lot. See Lot Size Units for the units of measurement (Square Feet, Square Meters, Acres, etc) |
| StateOrProvince | Text field containing the accepted postal abbreviation for the state or province | 
| ListingKeyNumeric | same as ``ListingKey`` column |

the remaining 9 columns ending with '.1' are removed since they are duplicates of existing columns

In [21]:
# drop columns
listings_clean = listings_clean.drop(columns = cols_to_remove)

In [22]:
print(listings_clean.shape)
print(listings_clean.columns)

(607724, 41)
Index(['OriginalListPrice', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'UnparsedAddress', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'ListAgentFullName', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'YearBuilt',
       'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories',
       'HighSchool', 'Levels', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'year_month', 'rate_30yr_fixed'],
      dtype='object')


### final shape after dropping: (607724, 41)

### CLEAN ROWS

now that we've dealt with columns, lets look into invalid values for ClosePrice, LivingArea, DOM, and negative bedrooms or bathrooms

In [23]:
# flag for negative closeprice
listings_clean['neg_closeprice_flag'] = listings_clean['ClosePrice'] <= 0

listings_clean[listings_clean['neg_closeprice_flag'] == True]

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag


In [24]:
# flag for negative livingarea
listings_clean['neg_livingarea_flag'] = listings_clean['LivingArea'] <= 0

listings_clean[listings_clean['neg_livingarea_flag'] == True]

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag
2949,6495000.0,1059501249,NaN,34.067738,-118.413460,605 Trenton Drive,0.0,6250000.0,91,Coldwell Banker Realty,...,False,3.0,NaN,90210,NaN,12898.0,2024-01,6.6425,False,True
6717,16000000.0,1058455780,NaN,34.089788,-118.466626,1111 Linda Flora Drive,0.0,16000000.0,39,The Agency,...,False,NaN,NaN,90049,NaN,74712.0,2024-01,6.6425,False,True
8819,498000.0,1058378873,NaN,NaN,NaN,1930 Mission St 304,0.0,498000.0,95,Mosaik Real Estate,...,False,1.0,NaN,94103,210.0,0.0,2024-01,6.6425,False,True
8852,17995000.0,1058377309,NaN,34.071606,-118.483070,735 N Bonhill Road,0.0,16995000.0,65,Carolwood Estates,...,False,NaN,NaN,90049,NaN,102588.0,2024-01,6.6425,False,True
9274,12950000.0,1058311519,NaN,34.081654,-118.936966,12815 Yellow Hill Road,0.0,12950000.0,70,Compass,...,False,6.0,NaN,90265,NaN,1393920.0,2024-01,6.6425,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
582932,259000.0,1159195492,NaN,33.705367,-116.264022,80394 Avenue 48 359,0.0,259000.0,0,LPT Realty,...,False,0.0,NaN,92201,670.0,3049.0,2026-05,6.4425,False,True
590854,475000.0,1175507140,NaN,35.398728,-118.907176,4900 Sweet Grass Drive,0.0,475000.0,8,eXp Realty of Greater Los Angeles,...,False,NaN,NaN,93306,NaN,10019.0,2026-06,6.4900,False,True
595365,21900000.0,1174763121,NaN,34.447811,-119.629516,1502 E Mountain Drive,0.0,21900000.0,8,Berkshire Hathaway HomeServices California Pro...,...,False,NaN,NaN,93108,NaN,92347.0,2026-06,6.4900,False,True
599084,39800000.0,1174197337,NaN,34.094369,-118.383191,1302 Collingwood Place,0.0,39800000.0,8,"Douglas Elliman of California, Inc.",...,False,8.0,NaN,90069,NaN,20803.0,2026-06,6.4900,False,True


In [25]:
# look @ DOM
listings_clean[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

,ListingContractDate,PurchaseContractDate,DaysOnMarket
0,2024-01-01,2024-05-07,127
1,2024-01-24,NaT,1
2,2024-01-12,NaT,1
3,2024-01-20,NaT,0
4,2024-01-12,NaT,2


In [26]:
listings_clean['PurchaseContractDate'].isna().sum()

np.int64(300700)

In [27]:
# flag @ negative DOM
listings_clean['neg_dom_flag'] = listings_clean['DaysOnMarket'] < 0

print(listings_clean[listings_clean['neg_dom_flag'] == True].shape)
listings_clean[listings_clean['neg_dom_flag'] == True][['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

(32, 44)


,ListingContractDate,PurchaseContractDate,DaysOnMarket
162,2024-01-30,2024-02-02,-48
166,2024-01-11,2024-01-22,-58
873,2024-01-31,2024-02-06,-1
2725,2024-01-25,NaT,-33
12371,2024-01-09,2024-02-27,-3


In [28]:
# clean negative dom

In [29]:
# look @ negative bedrooms
print(listings_clean[listings_clean['BedroomsTotal'] < 0].shape)
listings_clean[listings_clean['BedroomsTotal'] < 0].head()

(0, 44)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag


In [30]:
# look @ negative bathrooms
listings_clean[listings_clean['BathroomsTotalInteger'] < 0]

,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag


### data consistency checks
- validate logical order of date fields (ListingContractDate should precede PurchaseContractDate which should precede CloseDate)
- create boolean flag cols to mark records that violate these rules
    - listing_after_close_flag
    - purchase_after_close_flag
    - negative_timeline_flag

In [31]:
# validate order of datefields
listings_clean['negative_timeline_flag'] = listings_clean['ListingContractDate'] > listings_clean['PurchaseContractDate']

print('Shape where date fields are out of order', listings_clean[listings_clean['negative_timeline_flag'] == True].shape)
listings_clean[listings_clean['negative_timeline_flag'] == True][['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head()

Shape where date fields are out of order (305, 45)


,ListingContractDate,PurchaseContractDate,DaysOnMarket
1406,2024-01-31,2024-01-15,0
1712,2024-01-18,2024-01-05,0
2665,2024-01-29,2024-01-08,0
4140,2024-01-26,2024-01-10,0
12206,2024-01-11,2024-01-09,0


### geographic data checks
- flag records w missing coords (lat/lon is null)
- flag lat = 0 or lon = 0 (sentinel null vals)
- flag lon > 0 errors (cal coords should be negative)
- flag out-of-state/implausible coords

In [32]:
# flag missing coords (lat/lon is null)
listings_clean['missing_coords_flag'] = (listings_clean['Latitude'].isna()) | (listings_clean['Longitude'].isna())

print(listings_clean[listings_clean['missing_coords_flag'] == True].shape)
listings_clean[listings_clean['missing_coords_flag'] == True].head()

(80824, 46)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,PostalCode,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag
33,730000.0,1071104851,730000.0,NaN,NaN,2456 Havenscourt Blvd,1266.0,730000.0,35,Lifestyle Global Properties,...,94605,NaN,4000.0,2024-01,6.6425,False,False,False,False,True
46,950000.0,1069615513,NaN,NaN,NaN,1264 Camino Carmelo,2011.0,950000.0,11,"eXp Realty of Southern California, Inc.",...,91913,139.0,3374.0,2024-01,6.6425,False,False,False,False,True
53,1800000.0,1068189553,1714000.0,NaN,NaN,139 Ardith Drive,1633.0,1800000.0,39,Compass,...,94563,NaN,13900.0,2024-01,6.6425,False,False,False,False,True
109,739000.0,1065137461,NaN,NaN,NaN,898 Cosmo Ave,1232.0,785000.0,7,Coldwell Banker Realty,...,92019,NaN,10000.0,2024-01,6.6425,False,False,False,False,True
126,874999.0,1064977753,NaN,NaN,NaN,812 S Pacific Street Unit #1,693.0,874999.0,22,"eXp Realty of California, Inc.",...,92054,350.0,4860.0,2024-01,6.6425,False,False,False,False,True


In [33]:
# lat = 0 or lon = 0
listings_clean['sentinel_coords_flag'] = (listings_clean['Latitude'] == 0) | (listings_clean['Longitude'] == 0)

print(listings_clean[listings_clean['sentinel_coords_flag'] == True].shape)
listings_clean[listings_clean['sentinel_coords_flag'] == True].head()

(75, 47)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,AssociationFee,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag
7401,529900.0,1058425091,NaN,0.0,0.0,819 Norfolk Avenue,1908.0,505000.0,88,New Palace Realty,...,0.0,5300.0,2024-01,6.6425,False,False,False,False,False,True
39239,399000.0,1064843666,NaN,0.0,0.0,25604 6th Street,800.0,399000.0,35,REALTY ONE GROUP WEST,...,0.0,5040.0,2024-03,6.8200,False,False,False,False,False,True
140430,679000.0,1077531250,NaN,0.0,0.0,5280 Beachfront Cove Street 239,1777.0,679000.0,14,The Real Estate Guys,...,443.0,250544.0,2024-07,6.8475,False,False,False,False,False,True
152169,872008.0,1082054167,NaN,0.0,0.0,1960 Marigold St.,2202.0,872008.0,3,KB Home Sales -Northern California Inc,...,NaN,7000.0,2024-08,6.5000,False,False,False,False,False,True
152252,946000.0,1082052229,NaN,0.0,0.0,1646 Siderits Way,1892.0,946000.0,1,KB Home Sales -Northern California Inc,...,155.0,3038.0,2024-08,6.5000,False,False,False,False,False,True


In [34]:
# lon > 0 errors (cali coords should be negative)
listings_clean['pos_lon_flag'] = listings_clean['Longitude'] > 0

print(listings_clean[listings_clean['pos_lon_flag'] == True].shape)
listings_clean[listings_clean['pos_lon_flag'] == True].head()

(92, 48)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,LotSizeSquareFeet,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag
5053,1225000.0,1058722773,NaN,35.118938,24.547506,10100 Triopetra crete Greece,2350.0,1225000.0,8,"Theodore Kyriazis, Realtors",...,0.0,2024-01,6.6425,False,False,False,False,False,False,True
37837,771000.0,1064940544,805000.0,33.852530,118.251830,1129 E Gladwick Street,1822.0,771000.0,6,Island Legacy Real Estate,...,4905.0,2024-03,6.8200,False,False,False,False,False,False,True
44535,1385000.0,1063434053,NaN,33.445500,117.065490,18265 Reata Way,1999.0,1385000.0,1,Century 21 Affiliated,...,10800.0,2024-03,6.8200,False,False,False,False,False,False,True
53621,2900000.0,1061958359,NaN,38.711494,23.050674,"0 Paradromos A/d Pathe, Livantes, Greece",12163.0,2900000.0,8,Coldwell Banker Realty,...,48760.0,2024-03,6.8200,False,False,False,False,False,False,True
55921,965000.0,1073169639,NaN,737.000000,5.000000,15321 San Ardo Drive,1682.0,965000.0,8,HOME SWEET HOME REALTY,...,7642.0,2024-04,6.9925,False,False,False,False,False,False,True


In [35]:
# out of state (oos) or implausible coords

'''cali coords range
lat: (32.0, 42.5)
lon: (-125.0, -113.5)
'''

# filter for only cali listings
listings_clean['oos_coords_flag'] = ~(listings_clean['Latitude'].between(32.0, 42.5) & listings_clean['Longitude'].between(-125.0, -113.5))

listings_clean['oos_coords_flag'].value_counts()

oos_coords_flag
False    526573
True      81151
Name: count, dtype: int64

In [36]:
print('# rows with missing coordinates:', listings_clean[listings_clean['missing_coords_flag'] == True].shape[0])
print('# rows with sentinel coordinates:', listings_clean[listings_clean['sentinel_coords_flag'] == True].shape[0])

print('# rows with non-California values (negative longitude):', listings_clean[listings_clean['pos_lon_flag'] == True].shape[0])
print('# rows with out of state/implausible coordinates:', listings_clean[listings_clean['oos_coords_flag'] == True].shape[0])

# rows with missing coordinates: 80824
# rows with sentinel coordinates: 75
# rows with non-California values (negative longitude): 92
# rows with out of state/implausible coordinates: 81151


In [37]:
listings_clean.shape

(607724, 49)

### review flagged cols

We created 8 flagged columns:
- neg_closeprice_flag
- neg_livingarea_flag
- neg_dom_flag
- negative_timeline_flag
- missing_coords_flag
- sentinel_coords_flag
- pos_lon_flag
- oos_coords_flag

So now we can start cleaning by rows. Note that listing_after_close_flag and purchase_after_close_flag were not created since this dataset does not have a closedate column

In [38]:
listings_clean.iloc[:,-8:]

,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
607719,False,False,False,False,False,False,False,False
607720,False,False,False,False,False,False,False,False
607721,False,False,False,False,False,False,False,False
607722,False,False,False,False,False,False,False,False


In [39]:
# look into negative closeprice values
print(listings_clean[listings_clean['neg_closeprice_flag'] == True].shape)
listings_clean[listings_clean['neg_closeprice_flag'] == True].head()

(0, 49)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag


In [40]:
# look into negative livingarea values
print(listings_clean[listings_clean['neg_livingarea_flag'] == True].shape)
print(listings_clean[listings_clean['neg_livingarea_flag'] == True]['LivingArea'].head())

listings_clean[listings_clean['neg_livingarea_flag'] == True]['LivingArea'].value_counts()

(393, 49)
2949    0.0
6717    0.0
8819    0.0
8852    0.0
9274    0.0
Name: LivingArea, dtype: float64


LivingArea
0.0    393
Name: count, dtype: int64

In [41]:
# remove 0-value livingarea
print('Shape before removing:', listings_clean.shape)
listings_clean = listings_clean[listings_clean['neg_livingarea_flag'] == False]

print('Shape after removing:', listings_clean.shape)
listings_clean.head(3)

Shape before removing: (607724, 49)
Shape after removing: (607331, 49)


,OriginalListPrice,ListingKey,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,ListPrice,DaysOnMarket,ListOfficeName,...,year_month,rate_30yr_fixed,neg_closeprice_flag,neg_livingarea_flag,neg_dom_flag,negative_timeline_flag,missing_coords_flag,sentinel_coords_flag,pos_lon_flag,oos_coords_flag
0,1340000.0,1074973329,NaN,34.052207,-118.408445,2220 Avenue Of The Stars 2704,1301.0,1340000.0,127,Rodeo Realty- Brentwood,...,2024-01,6.6425,False,False,False,False,False,False,False,False
1,2500000.0,1074954552,NaN,33.496363,-117.691677,16 Palisades,2788.0,2500000.0,1,Your Home Sold Guaranteed Realty,...,2024-01,6.6425,False,False,False,False,False,False,False,False
2,3150000.0,1074936537,NaN,34.119345,-118.111254,1615 Waverly Road,3250.0,3150000.0,1,COMPASS,...,2024-01,6.6425,False,False,False,False,False,False,False,False


In [42]:
# set negative dom values to null
print('# rows with negative DOM:', listings_clean[listings_clean['neg_dom_flag'] == True]['DaysOnMarket'].shape)

print('# nulls before conversion:', listings_clean['DaysOnMarket'].isna().sum())
listings_clean.loc[listings_clean['neg_dom_flag'] == True, 'DaysOnMarket'] = np.nan

print('# nulls after conversion:', listings_clean['DaysOnMarket'].isna().sum())

# rows with negative DOM: (32,)
# nulls before conversion: 0
# nulls after conversion: 32


In [43]:
# investigate negative timeline flag
mask = listings_clean[listings_clean['negative_timeline_flag'] == True]
print(mask.shape)

mask[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].head(3)

(304, 49)


,ListingContractDate,PurchaseContractDate,DaysOnMarket
1406,2024-01-31,2024-01-15,0.0
1712,2024-01-18,2024-01-05,0.0
2665,2024-01-29,2024-01-08,0.0


In [44]:
# turn wrong dates into null
print('# rows with wrong timeline:', mask[['ListingContractDate', 'PurchaseContractDate', 'DaysOnMarket']].shape)

print('\n# nulls before conversion:', listings_clean[['ListingContractDate', 'PurchaseContractDate']].isna().sum())
listings_clean.loc[listings_clean['negative_timeline_flag'] == True, ['ListingContractDate', 'PurchaseContractDate']] = np.nan

print('\n# nulls after conversion:', listings_clean[['ListingContractDate', 'PurchaseContractDate']].isna().sum())

# rows with wrong timeline: (304, 3)

# nulls before conversion: ListingContractDate          0
PurchaseContractDate    300413
dtype: int64

# nulls after conversion: ListingContractDate        304
PurchaseContractDate    300717
dtype: int64


In [45]:
# investigate missing coords flag
mask = listings_clean[listings_clean['missing_coords_flag'] == True]
print('# rows with missing coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

# rows with missing coordinates: 80802


,Latitude,Longitude,City,PostalCode
33,NaN,NaN,Oakland,94605
46,NaN,NaN,Chula Vista,91913
53,NaN,NaN,Orinda,94563
109,NaN,NaN,El Cajon,92019
126,NaN,NaN,Oceanside,92054


In [46]:
# investigate sentinel coords
mask = listings_clean[listings_clean['sentinel_coords_flag'] == True]
print('# rows with sentinel coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']].head()

# rows with sentinel coordinates: 75


,Latitude,Longitude,City,PostalCode
7401,0.0,0.0,Patterson,95363
39239,0.0,0.0,San Bernardino,92410
140430,0.0,0.0,San Diego,92154
152169,0.0,0.0,Hollister,95023
152252,0.0,0.0,Gilroy,95020


In [47]:
# remove 0 lat/lon values
print('Shape before removing:', listings_clean.shape)
listings_clean = listings_clean[listings_clean['sentinel_coords_flag'] == False]

print('Shape after removing:', listings_clean.shape)

Shape before removing: (607331, 49)
Shape after removing: (607256, 49)


In [48]:
# investigate positive longitude flag
mask = listings_clean[listings_clean['pos_lon_flag'] == True]
print('# rows outside CA:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

# rows outside CA: 92


,Latitude,Longitude,City,PostalCode
5053,35.118938,24.547506,NaN,0
37837,33.852530,118.251830,Carson,90746
44535,33.445500,117.065490,Rancho Bernardo (San Diego),92128
53621,38.711494,23.050674,NaN,90210
55921,737.000000,5.000000,La Mirada,90638
...,...,...,...,...
586788,33.958450,117.600560,Eastvale,92880
595964,35.405414,118.831230,Bakersfield,93306
596070,35.408930,119.108270,Bakersfield,93312
600264,47.317371,8.211500,NaN,0


In [49]:
# set coordinate values to null
# even though cities correspond to their zipcodes, the lat/lon aren't within CA bounds
print('Before conversion:')
print(listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    ['Latitude', 'Longitude']
].head())

# some coordinates seem to be misinputted, so we can flip longitude sign
listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    'Longitude'
] *= -1

print('\nAfter conversion:')
print(listings_clean.loc[
    listings_clean['pos_lon_flag'] == True,
    ['Latitude', 'Longitude']
].head())

Before conversion:
         Latitude   Longitude
5053    35.118938   24.547506
37837   33.852530  118.251830
44535   33.445500  117.065490
53621   38.711494   23.050674
55921  737.000000    5.000000

After conversion:
         Latitude   Longitude
5053    35.118938  -24.547506
37837   33.852530 -118.251830
44535   33.445500 -117.065490
53621   38.711494  -23.050674
55921  737.000000   -5.000000


In [50]:
# we can reflag the pos_lon_flag to see if the bool changes
print('Before reflagging:')
print(listings_clean['pos_lon_flag'].value_counts())

listings_clean['pos_lon_flag'] = listings_clean['Longitude'] > 0

print('\nAfter reflagging:')
print(listings_clean['pos_lon_flag'].value_counts())


Before reflagging:
pos_lon_flag
False    607164
True         92
Name: count, dtype: int64

After reflagging:
pos_lon_flag
False    607256
Name: count, dtype: int64


In [51]:
# also reflag oos coords
print('Before reflagging:')
print(listings_clean['oos_coords_flag'].value_counts())

listings_clean['oos_coords_flag'] = ~(listings_clean['Latitude'].between(32.0, 42.5) & listings_clean['Longitude'].between(-125.0, -113.5))

print('\nAfter reflagging:')
listings_clean['oos_coords_flag'].value_counts()

Before reflagging:
oos_coords_flag
False    526203
True      81053
Name: count, dtype: int64

After reflagging:


oos_coords_flag
False    526249
True      81007
Name: count, dtype: int64

In [52]:
# look into out of state coords
mask = listings_clean[listings_clean['oos_coords_flag'] == True]
print('# rows with out of state/implausible coordinates:', mask.shape[0])

mask[['Latitude', 'Longitude', 'City', 'PostalCode']]

# rows with out of state/implausible coordinates: 81007


,Latitude,Longitude,City,PostalCode
33,NaN,NaN,Oakland,94605
46,NaN,NaN,Chula Vista,91913
53,NaN,NaN,Orinda,94563
109,NaN,NaN,El Cajon,92019
126,NaN,NaN,Oceanside,92054
...,...,...,...,...
607549,NaN,NaN,Alpine,91901
607589,NaN,NaN,Pine Valley,91962
607640,NaN,NaN,Strawberry,95375
607659,NaN,NaN,Concord,94518


In [53]:
listings_clean[(listings_clean['oos_coords_flag'] == True) &
               ~(listings_clean['Latitude'].notnull() &
               listings_clean['Longitude'].notnull())
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

,Latitude,Longitude,City,PostalCode
33,NaN,NaN,Oakland,94605
46,NaN,NaN,Chula Vista,91913
53,NaN,NaN,Orinda,94563
109,NaN,NaN,El Cajon,92019
126,NaN,NaN,Oceanside,92054
...,...,...,...,...
607549,NaN,NaN,Alpine,91901
607589,NaN,NaN,Pine Valley,91962
607640,NaN,NaN,Strawberry,95375
607659,NaN,NaN,Concord,94518


In [54]:
# remove non-cali coords via lat/lon
mask = (
        listings_clean['oos_coords_flag'] == True &
        ~(
            listings_clean['Latitude'].notnull() &
            listings_clean['Longitude'].notnull()
        )
        )

print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)
listings_clean = listings_clean[mask]

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

Shape before removing: (607256, 49)
Shape of oos_cords_flag before removal: (607256,)
Shape after removing: (607051, 49)
Shape of oos_cords_flag after removal: (607051,)


In [55]:
listings_clean['oos_coords_flag'].value_counts()

oos_coords_flag
False    526249
True      80802
Name: count, dtype: int64

In [56]:
listings_clean[(listings_clean['oos_coords_flag'] == True) &
               (~listings_clean['PostalCode'].astype(str).str.startswith('9'))
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

,Latitude,Longitude,City,PostalCode
49581,NaN,NaN,NaN,88888
61494,NaN,NaN,NaN,88888
68901,NaN,NaN,NaN,88888
79373,NaN,NaN,Chico,05073
81969,NaN,NaN,Outside Area (Outside Ca),00000
108456,NaN,NaN,Outside Area (Outside Ca),81401
113978,NaN,NaN,Outside Area (Inside Ca),22634
117726,NaN,NaN,Outside Area (Outside Ca),22715
118048,NaN,NaN,Duarte,01010
122791,NaN,NaN,Outside Area (Outside U.S.) Foreign Country,22753


In [57]:
# remove non-cali via postal code
mask = (
    listings_clean['oos_coords_flag'] &
    ~listings_clean['PostalCode'].astype(str).str.startswith('9')
)

print("Rows to remove:", mask.sum())

print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)
listings_clean = listings_clean[~mask]

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

Rows to remove: 31
Shape before removing: (607051, 49)
Shape of oos_cords_flag before removal: (607051,)
Shape after removing: (607020, 49)
Shape of oos_cords_flag after removal: (607020,)


In [60]:
listings_clean[listings_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']].isna().sum()

Latitude      80771
Longitude     80771
City            177
PostalCode        0
dtype: int64

In [68]:
listings_clean[listings_clean['oos_coords_flag']
               & listings_clean['City'].isna()
               ]['PostalCode'].unique()

array(['92061', '94065', '95004', '93933', '95670', '92111', '95703',
       '91932', '94074', '93908', '91326', '91962', '95419', '94904',
       '92240', '91906', '94574', '95497', '95002', '92128', '94028',
       '96022', '90710', '92129', '96137', '92109', '93291', '95023',
       '99999', '94516', '95635', '95076', '95346', '92037', '92549',
       '92676', '93266', '95628', '95664', '91411', '95006', '92123',
       '94062', '93222', '94520-1105', '92075', '93015', '96136', '95246',
       '95545', '95961', '95614', '95364', '96048', '93608', '94303',
       '95385', '95364-000', '92024', '92028', '96146', '95305', '95314'],
      dtype=object)

In [70]:
# convert nan cities to the correct zipcode
zip_to_city = {
    '92061': 'Pauma Valley',
    '94065': 'Redwood City',
    '95004': 'Aromas',
    '93933': 'Marina',
    '95670': 'Rancho Cordova',
    '92111': 'San Diego',
    '95703': 'Auburn',
    '91932': 'Imperial Beach',
    '94074': 'San Gregorio',
    '93908': 'Salinas',
    '91326': 'Porter Ranch',
    '91962': 'Pine Valley',
    '95419': 'Camp Meeker',
    '94904': 'Greenbrae',
    '92240': 'Desert Hot Springs',
    '91906': 'Campo',
    '94574': 'Saint Helena',
    '95497': 'The Sea Ranch',
    '95002': 'Alviso',
    '92128': 'San Diego',
    '94028': 'Portola Valley',
    '96022': 'Cottonwood',
    '90710': 'Harbor City',
    '92129': 'San Diego',
    '96137': 'Westwood',
    '92109': 'San Diego',
    '93291': 'Visalia',
    '95023': 'Hollister',
    '99999': np.nan,  # invalid zipcode
    '94516': 'Crockett',
    '95635': 'Greenwood',
    '95076': 'Watsonville',
    '95346': 'Mi Wuk Village',
    '92037': 'La Jolla',
    '92549': 'Idyllwild',
    '92676': 'Silverado',
    '93266': 'Stratford',
    '95628': 'Fair Oaks',
    '95664': 'Pilot Hill',
    '91411': 'Van Nuys',
    '95006': 'Boulder Creek',
    '92123': 'San Diego',
    '94062': 'Redwood City',
    '93222': 'Frazier Park',
    '94520': 'Concord',
    '92075': 'Solana Beach',
    '93015': 'Fillmore',
    '96136': 'Susanville',
    '95246': 'Linden',
    '95545': 'Honeydew',
    '95961': 'Olivehurst',
    '95614': 'Cool',
    '95364': 'Stevinson',
    '96048': 'Junction City',
    '93608': 'Cantua Creek',
    '94303': 'Palo Alto',
    '95385': 'Winton',
    '92024': 'Encinitas',
    '92028': 'Fallbrook',
    '96146': 'Olympic Valley',
    '95305': 'Coulterville',
    '95314': 'Cressey'
}

In [71]:
# normalize zipcodes to remove the hyphens
listings_clean['PostalCode'] = (
    listings_clean['PostalCode']
    .astype(str)
    .str[:5]
)

# map zipcode to the correct city
listings_clean['City'] = (
    listings_clean['City']
    .fillna(
        listings_clean['PostalCode'].astype(str).map(zip_to_city)
    )
)

# confirm changes were made
listings_clean[listings_clean['oos_coords_flag'] == True][['Latitude', 'Longitude', 'City', 'PostalCode']].isna().sum()

Latitude      80771
Longitude     80771
City             14
PostalCode        0
dtype: int64

In [72]:
listings_clean[listings_clean['oos_coords_flag']
               & listings_clean['City'].isna()
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

,Latitude,Longitude,City,PostalCode
96215,NaN,NaN,NaN,99999
118361,NaN,NaN,NaN,99999
276126,NaN,NaN,NaN,99999
290584,NaN,NaN,NaN,99999
368011,NaN,NaN,NaN,99999
418101,NaN,NaN,NaN,99999
418103,NaN,NaN,NaN,99999
418106,NaN,NaN,NaN,99999
419737,NaN,NaN,NaN,99999
420683,NaN,NaN,NaN,99999


In [73]:
# remove rows with 99999 postal code
print('Shape before removing:', listings_clean.shape)
print('Shape of oos_cords_flag before removal:', listings_clean['oos_coords_flag'].shape)

listings_clean = listings_clean[listings_clean['PostalCode'] != '99999']

print('Shape after removing:', listings_clean.shape)
print('Shape of oos_cords_flag after removal:', listings_clean['oos_coords_flag'].shape)

Shape before removing: (607020, 49)
Shape of oos_cords_flag before removal: (607020,)
Shape after removing: (606998, 49)
Shape of oos_cords_flag after removal: (606998,)


In [75]:
listings_clean[listings_clean['oos_coords_flag'] == True
               ][['Latitude', 'Longitude', 'City', 'PostalCode']]

,Latitude,Longitude,City,PostalCode
33,NaN,NaN,Oakland,94605
46,NaN,NaN,Chula Vista,91913
53,NaN,NaN,Orinda,94563
109,NaN,NaN,El Cajon,92019
126,NaN,NaN,Oceanside,92054
...,...,...,...,...
607549,NaN,NaN,Alpine,91901
607589,NaN,NaN,Pine Valley,91962
607640,NaN,NaN,Strawberry,95375
607659,NaN,NaN,Concord,94518


### deliverable
- document every xformation made & why
- include:
    - before/after counts
    - dtype confirmations
    - date consistency flag counts
    - geographic data quality summary noting invalid coordinate records